In [2]:
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [12]:
import faiss, pandas as pd, numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

What subset of transformers models would be best to try?

In [59]:
MODELS = {
    "MiniLM-L6": "all-MiniLM-L6-v2",
    "mpnet": "all-mpnet-base-v2",
    "BERT-base": "bert-base-nli-mean-tokens",
    "RoBERTa": "all-roberta-large-v1"
}

In [57]:
model_name = MODELS["MiniLM-L6"]
model = SentenceTransformer(model_name)

How about a set of experiments with different number length too. So dataset1 includes numbers 0-10, dataset2 0-100, and so on.
Then, let's compare performance between number vs spell-out.
- I think it also had trouble with floats?


In [7]:
# Load the dataset
text_path = "ages.txt" # back in the pycharm environment it will have to change to data/ages.txt

sentences = []
with open(text_path, "r") as f:
    for line in f:
        sentences.append(line.strip())

print("Number of sentences:", len(sentences))
print(sentences[:10])

Number of sentences: 100
['My age is 91', 'My age is 94', 'My age is 27', 'My age is 85', 'My age is 67', 'My age is 84', 'My age is 71', 'My age is 85', 'My age is 61', 'My age is 78']


In [11]:
"""
Saving Embeddings with FAISS for later when the dataset is bigger.
embeddings = []
BATCH = 128
for i in tqdm(range(0, len(sentences), BATCH)):
  batch = sentences[i:i+BATCH]
  emb = model.encode(batch)
  embeddings.append(emb)

embeddings = np.vstack(embeddings)

index = faiss.IndexFlatL2(model.get_sentence_embedding_dimension())
index.add(embeddings)"""

100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


In [19]:
def failproof(query_age):
  return min(sentences, key=lambda x: abs(int(x.split()[-1]) - query_age))

In [52]:
def simplest_query(age:int, sentences):
  query_age = str(age)
  emb_query = model.encode(query_age, convert_to_tensor=True)
  emb_sentences = model.encode(sentences, convert_to_tensor=True)
  similarities = util.cos_sim(emb_query, emb_sentences)[0]
  #similarities = util.dot_score(emb_query, emb_sentences)[0]

  best_idx = int(similarities.argmax())
  return sentences[best_idx]

In [53]:
def compare(age, sentences):
  return simplest_query(age, sentences), failproof(age)

In [54]:
transformer, math = compare(70, sentences)
print("transformer: ", transformer)
print("math: ", math)
# first one gave 78, second 99.

transformer:  My age is 99
math:  My age is 71


query: 70
transformer: 78
math: 71
second transformer model: 99

In [56]:
transformer, math = compare(4, sentences)
print("transformer: ", transformer)
print("math: ", math)

transformer:  My age is 1
math:  My age is 5


In [18]:
print(simplest_query(70, sentences))

My age is 78


I'm thinking... in the pipeline of experiments... apart from different transformers, maybe distances to the embeddings can be tested? Like the method for each, like L2 and other distances ETC.

In [60]:
query_ages = [0, 2, 10, 30, 46, 70, 85, 99, 100]

results = []
for model_label, model_name in MODELS.items():
  print(f"Evaluating model: {model_label}")
  model = SentenceTransformer(model_name)
  for q in query_ages:
    tf_choice = simplest_query(q, sentences)
    math_choice = failproof(q)
    results.append({
        "model": model_label,
        "query age": q,
        "transformer choice": tf_choice,
        "Expected": math_choice
        })

Evaluating model: MiniLM-L6


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [63]:
df = pd.DataFrame(results)
df.to_csv("results.csv")

print(df.pivot_table(index="model", values="Expected", aggfunc="count"))
print(df.head(50))

           Expected
model              
BERT-base         9
MiniLM-L6         9
RoBERTa           9
mpnet             9
        model  query age transformer choice      Expected
0   MiniLM-L6          0        My age is 0   My age is 0
1   MiniLM-L6          2        My age is 2   My age is 2
2   MiniLM-L6         10       My age is 10  My age is 10
3   MiniLM-L6         30       My age is 30  My age is 30
4   MiniLM-L6         46       My age is 46  My age is 46
5   MiniLM-L6         70       My age is 73  My age is 71
6   MiniLM-L6         85       My age is 85  My age is 85
7   MiniLM-L6         99       My age is 99  My age is 99
8   MiniLM-L6        100       My age is 99  My age is 99
9       mpnet          0        My age is 0   My age is 0
10      mpnet          2        My age is 1   My age is 2
11      mpnet         10        My age is 0  My age is 10
12      mpnet         30        My age is 0  My age is 30
13      mpnet         46        My age is 0  My age is 46
14      mp